# База данных системы мониторинга микроклимата

Модель: комната → ESP32 → DHT22 → два канала. Измерения хранятся отдельными строками, время — UTC. Пороги имеют период действия, предупреждение связано с измерением и порогом.

Запускай ячейки сверху вниз в ядре **microclimate**. Повторное заполнение справочников не создаёт дубликаты. Схема находится в `schema.sql`, функции — в `db.py` и `repo.py`.

## 1. Импорты и подключение
Путь к базе определяется относительно `db.py`, независимо от рабочей папки Jupyter.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "db.py").is_file():
    project_root = project_root.parent
if not (project_root / "db.py").is_file():
    raise FileNotFoundError("Открой блокнот из папки microclimate или microclimate/notebooks.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from db import DB_PATH, get_conn, initialize_schema
from repo import initialize_reference_data

if "conn" in globals():
    conn.close()
conn = get_conn(create=True)
print("База:", DB_PATH)

## 2. Схема и начальные данные
Создаются только отсутствующие таблицы и справочные записи. Существующие измерения и пороги сохраняются. Границы 20–26 °C и 40–60 % — учебные настройки, не нормативы. Эта операция не изменяет структуру уже существующих таблиц.

In [ ]:
initialize_schema(conn)
with conn:
    initialize_reference_data(conn)
print("Схема и начальные данные подготовлены.")

## 3. Семь запросов к модели
Проверяем связи, историю, статистику, действующие пороги и предупреждения. Пустой список предупреждений допустим, если нарушений не было.

In [ ]:
from repo import utc_now

# Интервал демонстрации в UTC; при необходимости измени даты.
start = "2026-10-19T00:00:00.000Z"
end = "2026-10-20T00:00:00.000Z"
now = utc_now()

print("\n1. Комнаты и узлы")
rows = conn.execute("""
    SELECT room.name, room.area_m2, node.device_uid, node.period_ms FROM room INNER JOIN node ON node.room_id = room.id
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n2. Датчики и каналы")
rows = conn.execute("""
    SELECT node.device_uid, sensor.model, channel.id, channel.metric, channel.unit FROM node INNER JOIN sensor ON sensor.node_id = node.id INNER JOIN channel ON channel.sensor_id = sensor.id ORDER BY channel.id
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n3. Последнее значение каждого канала")
rows = conn.execute("""
    SELECT channel.metric, m.ts, m.value, channel.unit FROM measurement AS m INNER JOIN channel ON channel.id = m.channel_id WHERE m.ts = (SELECT MAX(m2.ts) FROM measurement AS m2 WHERE m2.channel_id = m.channel_id) ORDER BY channel.id
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n4. Измерения за интервал")
rows = conn.execute("""
    SELECT channel_id, ts, value FROM measurement WHERE ts >= ? AND ts < ? ORDER BY ts, channel_id
""", (start, end)).fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n5. Статистика за интервал")
rows = conn.execute("""
    SELECT channel.metric, COUNT(*) AS count, MIN(m.value) AS minimum, MAX(m.value) AS maximum, AVG(m.value) AS average FROM measurement AS m INNER JOIN channel ON channel.id = m.channel_id WHERE m.ts >= ? AND m.ts < ? GROUP BY channel.id, channel.metric ORDER BY channel.id
""", (start, end)).fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n6. Действующие пороги")
rows = conn.execute("""
    SELECT channel.metric, threshold.lower_bound, threshold.upper_bound, threshold.valid_from, threshold.valid_to FROM threshold INNER JOIN channel ON channel.id = threshold.channel_id WHERE valid_from <= ? AND (valid_to IS NULL OR ? < valid_to) ORDER BY channel.id
""", (now, now)).fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n7. Предупреждения с измерениями и границами")
rows = conn.execute("""
    SELECT channel.metric, alert.ts, measurement.value, threshold.lower_bound, threshold.upper_bound, alert.direction FROM alert INNER JOIN channel ON channel.id = alert.channel_id INNER JOIN measurement ON measurement.channel_id = alert.channel_id AND measurement.ts = alert.ts INNER JOIN threshold ON threshold.id = alert.threshold_id ORDER BY alert.ts DESC
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

## 4. Завершение
Закрываем соединение. Для повторного просмотра запусти блокнот с начала.

In [ ]:
conn.close()
print("Соединение закрыто.")